# Stage 2 — Bangla mDistilBERT
One validation-only reportable run. Test data remains locked.

In [ ]:
RUN_HEAVY = False
SEED_TO_RUN = 42
ALLOWED_SEEDS = (42, 123, 2026)
CONFIGS = {seed: f'configs/stage2_reportable_bn_mdistilbert_seed{seed}.json' for seed in ALLOWED_SEEDS}
import json, subprocess, sys
from pathlib import Path
def find_repo():
    for base in [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working')]:
        for candidate in [base, *base.glob('**/Capstone-Project')]:
            if (candidate / 'corrected_pipeline' / 'config.py').is_file(): return candidate.resolve()
    raise FileNotFoundError('Capstone-Project repository not found')
repo = find_repo()
if SEED_TO_RUN not in ALLOWED_SEEDS: raise ValueError(f'SEED_TO_RUN must be one of {ALLOWED_SEEDS}')
config_path = repo / CONFIGS[SEED_TO_RUN]
sys.path.insert(0, str(repo))
from corrected_pipeline.config import load_config
config = load_config(config_path)
assert config['dataset']['languages'] == ['bangla']
assert config['model']['architecture'] == 'mdistilbert_single_task'
assert config['execution']['evaluate_test'] is False and config['execution']['test_locked'] is True
assert config['execution']['allow_overwrite'] is False
assert not any('test' in key.lower() for key in config['dataset']['paths'])
print('status:', config['result_status'])
print('model:', config['model']['name'], '@', config['model']['revision'])
print('language: bangla')
print('seed:', config['training']['random_seed'])
print('effective_batch_size:', config['training']['batch_size'] * config['training']['gradient_accumulation'])
print('dataset_hashes:', json.dumps(config['dataset']['hashes'], indent=2))
print('output_directory:', config['execution']['output_directory'])
if not RUN_HEAVY:
    print('Preflight only; training disabled.')
else:
    subprocess.run([sys.executable, '-m', 'compileall', '-q', 'corrected_pipeline', 'tests_stage1a'], cwd=repo, check=True)
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests_stage1a', '-v'], cwd=repo, check=True)
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('A Kaggle GPU accelerator is required')
    subprocess.run([sys.executable, '-m', 'corrected_pipeline.runner', '--config', str(config_path)], cwd=repo, check=True)
